In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from pathlib import Path
import glob
import re

In [28]:
# for pdb_path in glob.glob("sampling/train_ref/*.pdb"):
#     pdb_num = int(re.search(r'(\d+).pdb', pdb_path).group(1))
#     mol = Chem.MolFromPDBFile(pdb_path)
#     Chem.MolToPDBFile(mol, f"sampling/test_{pdb_num}.pdb")

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from rdkit.Chem import rdMolTransforms
from rdkit.Chem import rdMolAlign

In [ ]:
def load_pdb(path):
    mol = Chem.MolFromPDBFile(path, removeHs=True, sanitize=False)
    if mol is None:
        print(f"Failed to load {path}")
        return None
    for atom in mol.GetAtoms():
        atom.SetChiralTag(rdchem.ChiralType.CHI_UNSPECIFIED)

    for bond in mol.GetBonds():
        bond.SetStereo(rdchem.BondStereo.STEREONONE)
    try:
        Chem.SanitizeMol(mol)
    except:
        print(f"Failed to sanitize {path}")
        return None
    return mol

def heavy_atom_rmsd(mol_ref, mol_gen):
    # Remove hydrogens for heavy-atom RMSD
    ref = Chem.RemoveHs(mol_ref)
    gen = Chem.RemoveHs(mol_gen)

    return rdMolAlign.GetBestRMS(ref, gen)

def bond_length_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    errs = []
    for b in mol_ref.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()

        a_ri = mol_ref.GetAtomWithIdx(i)
        a_rj = mol_ref.GetAtomWithIdx(j)
        a_gi = mol_gen.GetAtomWithIdx(i)
        a_gj = mol_gen.GetAtomWithIdx(j)

        # atom identity checks
        assert a_ri.GetAtomicNum() == a_gi.GetAtomicNum(), f"Atom {i} mismatch"
        assert a_rj.GetAtomicNum() == a_gj.GetAtomicNum(), f"Atom {j} mismatch"

        d_r = conf_r.GetAtomPosition(i).Distance(conf_r.GetAtomPosition(j))
        d_g = conf_g.GetAtomPosition(i).Distance(conf_g.GetAtomPosition(j))

        errs.append(abs(d_r - d_g))

    return float(np.mean(errs))


def angle(a, b, c):
    ba = a - b
    bc = c - b
    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)
    if norm_ba == 0 or norm_bc == 0:
        return 0.0  # Return 0 if vectors are zero (shouldn't happen in valid molecules)
    cos_angle = np.dot(ba, bc) / (norm_ba * norm_bc)
    # Clamp to [-1, 1] to avoid NaN from floating point precision errors
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

def bond_angle_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    errs = []
    for atom in mol_ref.GetAtoms():
        nbrs = [n.GetIdx() for n in atom.GetNeighbors()]
        if len(nbrs) < 2:
            continue
        for i in range(len(nbrs)):
            for j in range(i + 1, len(nbrs)):
                a, b, c = nbrs[i], atom.GetIdx(), nbrs[j]
                ar = angle(conf_r.GetAtomPosition(a),
                           conf_r.GetAtomPosition(b),
                           conf_r.GetAtomPosition(c))
                ag = angle(conf_g.GetAtomPosition(a),
                           conf_g.GetAtomPosition(b),
                           conf_g.GetAtomPosition(c))
                errs.append(abs(ar - ag))
    return float(np.mean(errs))

def torsion_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    torsions = []

    for bond in mol_ref.GetBonds():
        if bond.IsInRing():
            continue
        if bond.GetBondType() != Chem.BondType.SINGLE:
            continue

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        ai = mol_ref.GetAtomWithIdx(i)
        aj = mol_ref.GetAtomWithIdx(j)

        if ai.GetDegree() < 2 or aj.GetDegree() < 2:
            continue

        ni = sorted(a.GetIdx() for a in ai.GetNeighbors() if a.GetIdx() != j)
        nj = sorted(a.GetIdx() for a in aj.GetNeighbors() if a.GetIdx() != i)

        for a in ni:
            for d in nj:
                tr = rdMolTransforms.GetDihedralDeg(conf_r, a, i, j, d)
                tg = rdMolTransforms.GetDihedralDeg(conf_g, a, i, j, d)

                diff = abs(tr - tg) % 360
                diff = min(diff, 360 - diff)
                torsions.append(diff)

    return float(np.mean(torsions)) if torsions else 0.0


def heavy_atom_rmsf(mol_list, align_first=True):
    """
    Calculate RMSF (Root Mean Square Fluctuation) for heavy atoms across multiple conformations.
    
    Args:
        mol_list: List of RDKit molecule objects (should be the same molecule, different conformations)
        align_first: If True, align all conformations to the first one before calculating RMSF
    
    Returns:
        Average RMSF over all heavy atoms
    """
    if len(mol_list) < 2:
        return 0.0
    
    # Remove hydrogens from all molecules
    mols_noH = [Chem.RemoveHs(mol) for mol in mol_list]
    
    # Check that all molecules have the same number of atoms
    n_atoms = mols_noH[0].GetNumAtoms()
    if not all(mol.GetNumAtoms() == n_atoms for mol in mols_noH):
        print("Warning: Not all conformations have the same number of atoms")
        return None
    
    # Get coordinates for all conformations
    coords_list = []
    for mol in mols_noH:
        conf = mol.GetConformer()
        coords = np.array([conf.GetAtomPosition(i) for i in range(n_atoms)])
        coords_list.append(coords)
    
    # Stack coordinates: [n_confs, n_atoms, 3]
    coords_array = np.stack(coords_list, axis=0)
    
    # Optionally align all conformations to the first one
    if align_first:
        ref_coords = coords_array[0]
        aligned_coords = [ref_coords]  # First conformation is reference
        for i in range(1, coords_array.shape[0]):
            # Align conformation i to reference using Kabsch algorithm
            mol_ref = mols_noH[0]
            mol_to_align = mols_noH[i]
            rmsd = rdMolAlign.GetBestRMS(mol_ref, mol_to_align)
            # Get the aligned conformation
            mol_aligned = Chem.Mol(mol_to_align)
            rdMolAlign.AlignMol(mol_aligned, mol_ref)
            conf_aligned = mol_aligned.GetConformer()
            coords_aligned = np.array([conf_aligned.GetAtomPosition(j) for j in range(n_atoms)])
            aligned_coords.append(coords_aligned)
        coords_array = np.stack(aligned_coords, axis=0)
    
    # Calculate mean position for each atom across conformations
    mean_coords = np.mean(coords_array, axis=0)  # [n_atoms, 3]
    
    # Calculate RMSF for each atom: sqrt(mean((pos_i - mean_pos)^2))
    squared_deviations = np.sum((coords_array - mean_coords[None, :, :]) ** 2, axis=2)  # [n_confs, n_atoms]
    rmsf_per_atom = np.sqrt(np.mean(squared_deviations, axis=0))  # [n_atoms]
    
    # Return average RMSF over all heavy atoms
    return float(np.mean(rmsf_per_atom))



def mmff_energy(mol):
    # AllChem.MMFFOptimizeMolecule(mol)
    props = AllChem.MMFFGetMoleculeProperties(mol)
    ff = AllChem.MMFFGetMoleculeForceField(mol, props)
    return ff.CalcEnergy()


from rdkit.Chem import rdchem
from rdkit.Chem import rdmolops


def clash_count(mol, scale=0.75):
    conf = mol.GetConformer()
    pts = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])
    vdw = np.array([
        rdchem.GetPeriodicTable().GetRvdw(a.GetAtomicNum())
        for a in mol.GetAtoms()
    ])

    n = mol.GetNumAtoms()
    clashes = 0

    for i in range(n):
        if mol.GetAtomWithIdx(i).GetAtomicNum() == 1:
            continue
        for j in range(i + 1, n):
            if mol.GetAtomWithIdx(j).GetAtomicNum() == 1:
                continue

            path = rdmolops.GetShortestPath(mol, i, j)
            if path is not None and len(path) <= 4:
                continue

            if np.linalg.norm(pts[i] - pts[j]) < scale * (vdw[i] + vdw[j]):
                clashes += 1

    return clashes

In [7]:
import glob
use_train = False

if use_train:
    ref_pdb_glob = "sampling/geom_drugs_ref/*.pdb"
    suffix = "_train"
else:
    ref_pdb_glob = "sampling/qmugs_ref//*.pdb"
    suffix = "_test"

# ref_pdb_glob = "sampling/train_ref/*.pdb"

ref_smi_to_mol = {}
for ref_pdb_path in glob.glob(ref_pdb_glob):
    mol = load_pdb(ref_pdb_path)
    if mol is not None:
        smi = Chem.MolToSmiles(mol)
        smi = Chem.CanonSmiles(smi)
        ref_smi_to_mol[smi] = mol


heavy_atom_min_rmsd_list = []
bond_length_min_mae_list = []
bond_angle_min_mae_list = []
torsion_min_mae_list = []
rmsf_list = []
energy_list = []

sample_nums = list(range(8))
valid_mols = 0

for sample_num in sample_nums:
    mol_list = []
    heavy_atom_rmsd_list = []
    bond_length_mae_list = []
    bond_angle_mae_list = []
    torsion_mae_list = []
    # for pdb_path in glob.glob(f"sampling/geom_og{suffix}/samples/*.pdb"):
    for pdb_path in glob.glob(f"sampling/geom_identityRot_frame_384{suffix}/samples/*.pdb"):
    # for pdb_path in glob.glob(f"sampling/geom_identityRot_frame_384{suffix}_no_corrupt/samples/*.pdb"):
        path_sample_num = int(Path(pdb_path).stem.split('_')[1])
        # path_sample_num = int(Path(pdb_path).stem.split('_')[2])
        if sample_num != path_sample_num:
            continue
        # if Path(pdb_path).stem != 'sample_0_0':
        #     continue
        # if 'test' not in pdb_path:
            # continue

        mol = load_pdb(pdb_path)
        if mol is not None:
            smi = Chem.MolToSmiles(mol)
            smi = Chem.CanonSmiles(smi)
            ref_mol = ref_smi_to_mol.get(smi, None)
            # print('ref_mol', pdb_path, ref_mol)
            if ref_mol is not None:
                ref_mol = Chem.AddHs(ref_mol)
                mol = Chem.AddHs(mol)
                # print(pdb_path, smi)
                # print(clash_count(mol))
                # print(heavy_atom_rmsd(mol, ref_mol))
                # print(bond_length_mae(mol, ref_mol))
                # print(bond_angle_mae(mol, ref_mol))
                # print(torsion_mae(mol, ref_mol))
                heavy_atom_rmsd_list.append(heavy_atom_rmsd(mol, ref_mol))
                bond_length_mae_list.append(bond_length_mae(mol, ref_mol))
                bond_angle_mae_list.append(bond_angle_mae(mol, ref_mol))
                torsion_mae_list.append(torsion_mae(mol, ref_mol))
                mol_list.append(mol) 
                valid_mols += 1
                # energy_list.append(mmff_energy(mol) - mmff_energy(ref_mol))
            # Calculate RMSF across all replicate conformations for this sample
    if len(mol_list) >= 2:
        rmsf = heavy_atom_rmsf(mol_list, align_first=True)
        if rmsf is not None:
            rmsf_list.append(rmsf)
    if len(heavy_atom_rmsd_list) > 0:
        best_heavy_atom_rmsd = min(heavy_atom_rmsd_list)
        best_idx = heavy_atom_rmsd_list.index(best_heavy_atom_rmsd)
        heavy_atom_min_rmsd_list.append(best_heavy_atom_rmsd)
        bond_length_min_mae_list.append(bond_length_mae_list[best_idx])
        bond_angle_min_mae_list.append(bond_angle_mae_list[best_idx])
        torsion_min_mae_list.append(torsion_mae_list[best_idx])


In [6]:
print(np.mean(heavy_atom_min_rmsd_list), len(heavy_atom_min_rmsd_list))
print(np.mean(bond_length_min_mae_list))
print(np.mean(bond_angle_min_mae_list))
print(np.mean(torsion_min_mae_list))
print(np.mean(rmsf_list), len(rmsf_list))
print(valid_mols)

2.573922289284227 6
1.4338686254028374
28.10381207471521
78.44227341558711
3.049916624015106 6
29


In [8]:
print(np.mean(heavy_atom_min_rmsd_list), len(heavy_atom_min_rmsd_list))
print(np.mean(bond_length_min_mae_list))
print(np.mean(bond_angle_min_mae_list))
print(np.mean(torsion_min_mae_list))
print(np.mean(rmsf_list), len(rmsf_list))
print(valid_mols)

2.7021745994093664 6
1.340904377797802
26.386928890943164
82.09132635161676
2.502153395302939 4
15
